# EXP-011 판정 근거: 다중 크기 청킹(512 + 256)

SPEC 판정(dev-full Recall@5)과 별개로, 보류일 때의 예외 채택 조건(Recall@1·MRR@10 95% 구간 하한 > 0, EXP-011 문서 "판정 기준")과 운영 부담(인덱스 용량·메모리·검색 시간)을 본다. 부트스트랩은 SPEC 판정과 같은 회의 단위 paired bootstrap(`src/rag/eval/bootstrap.py`, `configs/eval/spec_v1.yaml`)이다.

In [6]:
from pathlib import Path

import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run

In [7]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
INDEX = ROOT / 'data' / 'index'
BASE = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-dev-full.csv'
CAND = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-full.csv'
INDEX_DIRS = ['snowflake-ko-fixed-512-64', 'snowflake-ko-fixed-256-32']
DIM = 1024

## 지표별 차이와 95% 구간

In [8]:
cfg = load_cfg()
ev, bs = cfg.eval, cfg.eval.bootstrap
gold = load_gold(cfg, 'dev-full')
n_min = min_docs(ev.recall_ks, ev.mrr_k)
scores = {name: per_query(gold_ranks(read_run(RUNS / f), gold, n_min), ev.recall_ks, ev.mrr_k)
          for name, f in [('base', BASE), ('cand', CAND)]}
print(len(gold), 'queries', list(scores['base'].columns))

3016 queries ['recall@1', 'recall@5', 'recall@10', 'mrr@10']


In [9]:
rows = []
for m in scores['base'].columns:
    d = scores['cand'][m] - scores['base'][m]
    point, lo, hi = paired_bootstrap(d.to_numpy(), gold[bs.unit].to_numpy(), bs.n_resamples,
                                     bs.seed, bs.alpha)
    rows.append({'metric': m, 'base': scores['base'][m].mean(), 'cand': scores['cand'][m].mean(),
                 'diff_%p': point * 100, 'lo_%p': lo * 100, 'hi_%p': hi * 100, 'lo>0': lo > 0})
print(pd.DataFrame(rows).set_index('metric').round(4))

             base    cand  diff_%p   lo_%p   hi_%p  lo>0
metric                                                  
recall@1   0.8103  0.8369   2.6525  1.7821  3.5553  True
recall@5   0.9214  0.9337   1.2268  0.7425  1.7323  True
recall@10  0.9483  0.9572   0.8952  0.5267  1.2866  True
mrr@10     0.8602  0.8815   2.1238  1.5606  2.7253  True


## 질의별 이동

In [10]:
for m in ('recall@1', 'recall@5'):
    b, c = scores['base'][m], scores['cand'][m]
    print(f'{m}: 진입 {int(((b == 0) & (c == 1)).sum())}, 이탈 {int(((b == 1) & (c == 0)).sum())}')

recall@1: 진입 125, 이탈 45
recall@5: 진입 51, 이탈 14


## 인덱스 용량 (full)

임베딩은 float32 npy로 저장한다. 검색 때 dense 행렬은 512와 256 청크를 이어 붙여 메모리에 올린다. float16 열은 저장·메모리를 반으로 줄였을 때의 계산값이다(측정 아님).

In [11]:
def du(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

rows = []
for name in INDEX_DIRS:
    d = INDEX / name / 'full'
    n = sum(1 for _ in (d / 'chunks.jsonl').open(encoding='utf-8'))
    rows.append({'index': name, 'chunks': n, 'emb_MiB': du(d / 'emb') / 2**20,
                 'chunks_jsonl_MiB': (d / 'chunks.jsonl').stat().st_size / 2**20,
                 'emb_f16_MiB': n * DIM * 2 / 2**20})
sizes = pd.DataFrame(rows).set_index('index')
sizes.loc['합계'] = sizes.sum()
print(sizes.round(1))
base_n = sizes.loc[INDEX_DIRS[0], 'chunks']
print('dense 벡터 수', int(base_n), '→', int(sizes.loc['합계', 'chunks']),
      f"({sizes.loc['합계', 'chunks'] / base_n:.2f}배)")

                             chunks  emb_MiB  chunks_jsonl_MiB  emb_f16_MiB
index                                                                      
snowflake-ko-fixed-512-64  118039.0    461.1             252.6        230.5
snowflake-ko-fixed-256-32  220645.0    861.9             266.6        430.9
합계                         338684.0   1323.0             519.2        661.5
dense 벡터 수 118039 → 338684 (2.87배)


## 순위 이동의 출처

1위·5위 안으로 들어온 정답이 기준에서 몇 위였는지 본다. dev-small에서는 1위 진입 26건이 모두 기준 2\~5위였다(20_01).

In [13]:
rb = gold_ranks(read_run(RUNS / BASE), gold, n_min).fillna(99)
rc = gold_ranks(read_run(RUNS / CAND), gold, n_min).fillna(99)
for k in (1, 5):
    up, down = (rb > k) & (rc <= k), (rb <= k) & (rc > k)
    bins = {1: [1, 5, 10, 100], 5: [5, 10, 100]}[k]
    labels = {1: ['2~5위', '6~10위', '10위 밖'], 5: ['6~10위', '10위 밖']}[k]
    src = pd.cut(rb[up], bins, labels=labels).value_counts(sort=False)
    print(f'top{k} 진입 {int(up.sum())} (기준 순위: {src.to_dict()}), 이탈 {int(down.sum())}')

top1 진입 125 (기준 순위: {'2~5위': 101, '6~10위': 12, '10위 밖': 12}), 이탈 45
top5 진입 51 (기준 순위: {'6~10위': 29, '10위 밖': 22}), 이탈 14


## dense 단독 비교 (참고)

hybrid 결합 전 dense 목록만으로 256 청크 추가의 효과를 본다. 판정에 쓰지 않는다.

In [14]:
dense = {name: per_query(gold_ranks(read_run(RUNS / f), gold, n_min), ev.recall_ks, ev.mrr_k)
         for name, f in [('dense 512', 'snowflake-ko-fixed-512-64-dev-full.csv'),
                         ('dense 512+256', 'snowflake-ko-fixed-512-64-dense-ms256-dev-full.csv')]}
for m in ('recall@1', 'recall@5', 'mrr@10'):
    d = dense['dense 512+256'][m] - dense['dense 512'][m]
    point, lo, hi = paired_bootstrap(d.to_numpy(), gold[bs.unit].to_numpy(), bs.n_resamples,
                                     bs.seed, bs.alpha)
    print(f"{m}: {dense['dense 512'][m].mean():.4f} -> {dense['dense 512+256'][m].mean():.4f}, "
          f'{point * 100:+.2f}%p [{lo * 100:+.2f}, {hi * 100:+.2f}]')

recall@1: 0.7407 -> 0.7918, +5.11%p [+3.96, +6.24]
recall@5: 0.8810 -> 0.9092, +2.82%p [+1.98, +3.69]
mrr@10: 0.8009 -> 0.8436, +4.27%p [+3.46, +5.08]
